# Milestone 4 - Task 5: Email Body Feature Extraction

**Owner:** Klara Haxhiaj  
**Objective:** Extract 5 numerical body features that signal phishing intent: urgency language, link count, HTML ratio, word count, and average word length.

### Why we use the RAW cleaned text (not normalized)
Features are computed on `emails_clean.csv` (`text_combined`), **not** on M4-T2's normalized output. Normalization strips HTML tags and URLs, which would force `html_ratio` and `link_count` to always be zero. So this task reads directly from the M4-T1 output.

## Step 0 — Download input from S3 (run once in terminal)
```bash
mkdir -p data/processed
aws s3 cp s3://email-security-pipeline-datasets/datasets/processed/emails_clean.csv \
  data/processed/emails_clean.csv --profile lab-user
```

In [3]:
from pathlib import Path
import csv, re
csv.field_size_limit(10_000_000)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
INPUT_PATH  = PROJECT_ROOT / 'data' / 'processed' / 'emails_clean.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'emails_body_features.csv'
print('Input :', INPUT_PATH)
print('Output:', OUTPUT_PATH)

Input : <PROJECT_ROOT>/data/processed/emails_clean.csv
Output: <PROJECT_ROOT>/data/processed/emails_body_features.csv


## Step 1 — Define the 5 feature functions

In [ ]:
URGENCY_WORDS = [
    'urgent','immediately','verify','suspended','limited','action required',
    'click here','confirm','account','password','update','expire',
    'unusual activity','security alert',
]

def urgency_score(text):
    t = str(text).lower()
    return sum(1 for w in URGENCY_WORDS if w in t)

def count_links(text):
    return len(re.findall(r'https?://[^\s]+', str(text)))

def html_ratio(text):
    t = str(text)
    tags = len(re.findall(r'<[^>]+>', t))
    return round(tags / len(t), 4) if t else 0.0

def word_count(text):
    return len(str(text).split())

def avg_word_length(text):
    words = str(text).split()
    return round(sum(len(w) for w in words) / len(words), 2) if words else 0.0

## Step 2 — Load the dataset

In [ ]:
with INPUT_PATH.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))
print('Columns:', list(rows[0].keys()))
print('Total rows:', len(rows))

## Step 3 — Extract features per email

In [ ]:
out_rows = []
for r in rows:
    text = r['text_combined']
    out_rows.append({
        'label':           r['label'],
        'urgency_score':   urgency_score(text),
        'link_count':      count_links(text),
        'html_ratio':      html_ratio(text),
        'word_count':      word_count(text),
        'avg_word_length': avg_word_length(text),
    })

print('Feature columns:', [c for c in out_rows[0] if c != 'label'])
print('Sample row:', out_rows[0])

## Step 4 — Save the body feature matrix

In [ ]:
cols = ['label','urgency_score','link_count','html_ratio','word_count','avg_word_length']
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader(); w.writerows(out_rows)
print(f'Saved {len(out_rows)} rows, 5 features -> {OUTPUT_PATH}')

## Step 5 — Upload to S3 (run in terminal; do NOT commit the CSV)
```bash
aws s3 cp data/processed/emails_body_features.csv \
  s3://email-security-pipeline-datasets/datasets/processed/emails_body_features.csv \
  --profile lab-user
```